In [15]:
# Data Cleaning Project
## OASIS INFOBYTE — Data Analytics Internship
### L1 Task 3 — Cleaning Data

In [16]:
import pandas as pd
import numpy as np

In [17]:
df=pd.read_csv(r"C:\Users\Admin\Desktop\DATA ANALYTICS\SQL\GPT-SQL\Messy_Employee_dataset.csv")
df.head()

,Employee_ID,First_Name,Last_Name,Age,Department_Region,Status,Join_Date,Salary,Email,Phone,Performance_Score,Remote_Work
0,EMP1000,Bob,Davis,25.0,DevOps-California,Active,4/2/2021,59767.65,bob.davis@example.com,-1651623197,Average,True
1,EMP1001,Bob,Brown,NaN,Finance-Texas,Active,7/10/2020,65304.66,bob.brown@example.com,-1898471390,Excellent,True
2,EMP1002,Alice,Jones,NaN,Admin-Nevada,Pending,12/7/2023,88145.90,alice.jones@example.com,-5596363211,Good,True
3,EMP1003,Eva,Davis,25.0,Admin-Nevada,Inactive,11/27/2021,69450.99,eva.davis@example.com,-3476490784,Good,True
4,EMP1004,Frank,Williams,25.0,Cloud Tech-Florida,Active,1/5/2022,109324.61,frank.williams@example.com,-1586734256,Poor,False


In [18]:
pip install

Note: you may need to restart the kernel to use updated packages.


ERROR: You must give at least one requirement to install (see "pip help install")


In [19]:
df.shape


(1020, 12)

In [20]:
df.isnull().sum()

Employee_ID            0
First_Name             0
Last_Name              0
Age                  211
Department_Region      0
Status                 0
Join_Date              0
Salary                24
Email                  0
Phone                  0
Performance_Score      0
Remote_Work            0
dtype: int64

In [21]:
# --- Data Quality Report ---
print("=== DATA QUALITY REPORT ===")
print(f"Total Rows: {df.shape[0]}")
print(f"Total Columns: {df.shape[1]}")
print("\n--- Null Counts per Column ---")
print(df.isnull().sum())
print(f"\n--- Duplicate Rows Count: {df.duplicated().sum()} ---")
print("\n--- Data Types ---")
print(df.dtypes)
print("\n--- Value Range & Anomaly Check ---")
print(df.describe(include='all'))

=== DATA QUALITY REPORT ===
Total Rows: 1020
Total Columns: 12

--- Null Counts per Column ---
Employee_ID            0
First_Name             0
Last_Name              0
Age                  211
Department_Region      0
Status                 0
Join_Date              0
Salary                24
Email                  0
Phone                  0
Performance_Score      0
Remote_Work            0
dtype: int64

--- Duplicate Rows Count: 0 ---

--- Data Types ---
Employee_ID           object
First_Name            object
Last_Name             object
Age                  float64
Department_Region     object
Status                object
Join_Date             object
Salary               float64
Email                 object
Phone                  int64
Performance_Score     object
Remote_Work             bool
dtype: object

--- Value Range & Anomaly Check ---
       Employee_ID First_Name Last_Name         Age Department_Region  \
count         1020       1020      1020  809.000000              10

In [22]:
# Capture 'before' metrics
null_count_before = df.isnull().sum().sum()
row_count_before = len(df)
dup_count_before = df.duplicated().sum()

# 1. Age: Impute missing values with the median to avoid outlier skew
df['Age'] = df['Age'].fillna(df['Age'].median())

# 2. Salary: Impute missing values with the median salary
df['Salary'] = df['Salary'].fillna(df['Salary'].median())

In [23]:

df['Salary'].isnull().sum()

np.int64(0)

In [24]:
# Check and drop any duplicate rows
dup_count_before_dedup = df.duplicated().sum()
df = df.drop_duplicates().reset_index(drop=True)
print(f"Removed {dup_count_before_dedup} duplicate rows.")

Removed 0 duplicate rows.


In [25]:
# 1. Standardize Join_Date to datetime format
df['Join_Date'] = pd.to_datetime(df['Join_Date'], errors='coerce')

# 2. Fix anomalous negative phone numbers by taking absolute value
df['Phone'] = df['Phone'].abs()

# 3. Strip whitespace from string columns
for col in ['First_Name', 'Last_Name', 'Department_Region', 'Status', 'Performance_Score']:
    df[col] = df[col].astype(str).str.strip()

In [31]:
def handle_outliers_iqr(df, column):
    Q1 = df[column].quantile(0.25)
    Q3 = df[column].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    
    # Cap outliers to retain valuable records
    df[column] = np.where(df[column] < lower_bound, lower_bound, df[column])
    df[column] = np.where(df[column] > upper_bound, upper_bound, df[column])
    return df 
df = handle_outliers_iqr(df, 'Salary')
df = handle_outliers_iqr(df, 'Age')


In [32]:
df['Employee_ID'] = df['Employee_ID'].astype(str)
df['First_Name'] = df['First_Name'].astype(str)
df['Last_Name'] = df['Last_Name'].astype(str)
df['Age'] = df['Age'].astype(int)
df['Salary'] = df['Salary'].astype(float)
df['Phone'] = df['Phone'].astype(str) # Stored as string to preserve leading digits/formatting
df['Status'] = df['Status'].astype('category')
df['Performance_Score'] = df['Performance_Score'].astype('category')
df['Remote_Work'] = df['Remote_Work'].astype(bool)

print("\n--- Corrected Data Types ---")
print(df.dtypes)


--- Corrected Data Types ---
Employee_ID                  object
First_Name                   object
Last_Name                    object
Age                           int64
Department_Region            object
Status                     category
Join_Date            datetime64[ns]
Salary                      float64
Email                        object
Phone                        object
Performance_Score          category
Remote_Work                    bool
dtype: object


In [33]:
null_count_after = df.isnull().sum().sum()
row_count_after = len(df)
dup_count_after = df.duplicated().sum()

summary_table = pd.DataFrame({
    'Metric': ['Total Rows', 'Null Count', 'Duplicate Count', 'Data Type Accuracy'],
    'Before Cleaning': [row_count_before, null_count_before, dup_count_before, 'Inconsistent (Dates as string, negative phone int)'],
    'After Cleaning': [row_count_after, null_count_after, dup_count_after, '100% Correct dtypes']
})

print(summary_table.to_markdown(index=False))

| Metric             | Before Cleaning                                    | After Cleaning      |
|:-------------------|:---------------------------------------------------|:--------------------|
| Total Rows         | 1020                                               | 1020                |
| Null Count         | 235                                                | 0                   |
| Duplicate Count    | 0                                                  | 0                   |
| Data Type Accuracy | Inconsistent (Dates as string, negative phone int) | 100% Correct dtypes |


In [34]:
output_filename = 'cleaned_employee_dataset.csv'
df.to_csv(output_filename, index=False)
print(f"Cleaned dataset successfully saved to '{output_filename}'.")

Cleaned dataset successfully saved to 'cleaned_employee_dataset.csv'.
